<a href="https://colab.research.google.com/github/gunvarawork-cpu/code-repository/blob/main/LAB_2/LAB2_Spatial_Analysis_6606614623.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.5 MB/s eta 0:00:00


In [2]:
import ee, geemap
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-gunvarawoon')

In [5]:
# =========================
# 1) Study Area (Nan)
# =========================
admin1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
nan = admin1.filter(ee.Filter.eq('ADM1_NAME', 'Nan'))

admin2 = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Nan'))

# =========================
# 2) Load Sentinel-2 + Filter
# =========================
def maskS2(image):
    scl = image.select('SCL')
    cloud_mask = scl.neq(3) \
                    .And(scl.neq(8)) \
                    .And(scl.neq(9)) \
                    .And(scl.neq(10))
    return image.updateMask(cloud_mask).divide(10000)

def getCollection(start, end):
    collection = (ee.ImageCollection("COPERNICUS/S2_SR")
                  .filterBounds(nan)
                  .filterDate(start, end)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

    return collection.map(maskS2).select(['B3', 'B4', 'B8', 'B11'])

# Dry vs Wet
dry = getCollection('2023-02-01', '2023-04-30')
wet = getCollection('2023-11-01', '2023-12-31')

dry_img = dry.median().clip(nan)
wet_img = wet.median().clip(nan)

# =========================
# 3) Calculate Index
# =========================
def addIndex(img):
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
    ndbi = img.normalizedDifference(['B11','B8']).rename('NDBI')
    return img.addBands([ndvi, ndwi, ndbi])

dry_img = addIndex(dry_img)
wet_img = addIndex(wet_img)

# =========================
# 4) Visualization
# =========================
Map = geemap.Map()
Map.centerObject(nan, 8)

Map.add_basemap('SATELLITE')

ndvi_vis = {'min':0.15, 'max':0.85, 'palette':['white','green']}
ndwi_vis = {'min':-0.5, 'max':0.5, 'palette':['red','white','blue']}
change_vis = {'min':-0.2,'max':0.2,'palette':['red','yellow','green']}

Map.addLayer(dry_img.select('NDVI'), ndvi_vis, '🌿 NDVI Dry')
Map.addLayer(wet_img.select('NDVI'), ndvi_vis, '🌿 NDVI Wet')
Map.addLayer(dry_img.select('NDWI'), ndwi_vis, '💧 NDWI Dry')

# =========================
# 5) Change Detection
# =========================
ndvi_change = wet_img.select('NDVI').subtract(dry_img.select('NDVI'))
Map.addLayer(ndvi_change, change_vis, '🔄 NDVI Change')

# =========================
# 6) Zonal Statistics
# =========================
zonal = dry_img.select('NDVI').reduceRegions(
    collection=admin2,
    reducer=ee.Reducer.mean(),
    scale=30
)

zonal_df = pd.DataFrame([f['properties'] for f in zonal.getInfo()['features']])
print("Zonal Statistics (NDVI):")
print(zonal_df.head())

# =========================
# 7) Correlation
# =========================
sample = dry_img.select(['NDVI','NDWI']).sample(
    region=nan,
    scale=30,
    numPixels=3000
)

df = pd.DataFrame([f['properties'] for f in sample.getInfo()['features']])
print("\nCorrelation NDVI vs NDWI:")
print(df.corr())

# =========================
# 8) Area Calculation
# =========================
dense = dry_img.select('NDVI').gt(0.5)
water = dry_img.select('NDWI').gt(0.2)

area_dense = dense.multiply(ee.Image.pixelArea())
area_water = water.multiply(ee.Image.pixelArea())

dense_km2 = area_dense.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=nan,
    scale=30,
    maxPixels=1e13
).get('NDVI').getInfo() / 1e6

water_km2 = area_water.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=nan,
    scale=30,
    maxPixels=1e13
).get('NDWI').getInfo() / 1e6

print("\nDense vegetation area (km2):", dense_km2)
print("Water area (km2):", water_km2)

# =========================
# 9) show Map
# =========================
Map.addLayerControl()
Map

Zonal Statistics (NDVI):
   ADM0_CODE ADM0_NAME  ADM1_CODE ADM1_NAME  ADM2_CODE     ADM2_NAME  \
0        240  Thailand       2881       Nan      26756     Ban Luang   
1        240  Thailand       2881       Nan      26757        Bo Kua   
2        240  Thailand       2881       Nan      26758  Chiang Klang   
3        240  Thailand       2881       Nan      26759    Mae Charim   
4        240  Thailand       2881       Nan      26760     Muang Nan   

  DISP_AREA  EXP2_YEAR        STATUS  STR2_YEAR  Shape_Area  Shape_Leng  \
0        NO       3000  Member State       1000    0.040672    0.983181   
1        NO       3000  Member State       1000    0.106244    2.187518   
2        NO       3000  Member State       1000    0.027656    0.862821   
3        NO       3000  Member State       1000    0.080789    1.289580   
4        NO       3000  Member State       1000    0.121585    1.736030   

       mean  
0  0.357241  
1  0.381606  
2  0.311727  
3  0.371397  
4  0.324589  

Correl

Map(center=[18.847712206673826, 100.83316508361675], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# =========================
# 10) Export
# =========================

region = nan.geometry()

# =========================
# Create Tasks
# =========================
task1 = ee.batch.Export.image.toDrive(
    image=dry_img.select('NDVI'),
    description='NDVI_Dry',
    folder='GEE',
    fileNamePrefix='NDVI_Dry',
    region=region,
    scale=30,
    crs='EPSG:32647',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task2 = ee.batch.Export.image.toDrive(
    image=wet_img.select('NDVI'),
    description='NDVI_Wet',
    folder='GEE',
    fileNamePrefix='NDVI_Wet',
    region=region,
    scale=30,
    crs='EPSG:32647',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task3 = ee.batch.Export.image.toDrive(
    image=ndvi_change,
    description='NDVI_Change',
    folder='GEE',
    fileNamePrefix='NDVI_Change',
    region=region,
    scale=30,
    crs='EPSG:32647',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task4 = ee.batch.Export.image.toDrive(
    image=dry_img.select('NDWI'),
    description='NDWI_Dry',
    folder='GEE',
    fileNamePrefix='NDWI_Dry',
    region=region,
    scale=30,
    crs='EPSG:32647',
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

# =========================
# Start All Tasks
# =========================
tasks = [task1, task2, task3, task4]

for t in tasks:
    t.start()